<div style="background: #0f172a; padding: 40px 32px; border-radius: 12px; margin-bottom: 24px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

<p style="color: #94a3b8; font-size: 12px; letter-spacing: 0.18em; text-transform: uppercase; font-weight: 500; margin: 0 0 20px 0;">
  FINANCIAL CONTEXT &middot; GLOSSARY &middot; NEW TO FINANCE?
</p>

<h1 style="font-family: Georgia, 'Times New Roman', serif; font-size: 44px; line-height: 1.1; color: #f8fafc; font-weight: 400; margin: 0 0 8px 0;">
  Read the <em style="color: #5eead4; font-style: italic;">Glossary</em> first
</h1>

<p style="color: #cbd5e1; font-size: 16px; line-height: 1.6; margin: 24px 0 0 0; max-width: 720px;">
  This notebook assumes familiarity with concepts like <span style="color: #f8fafc; font-weight: 500;">10-K filings</span>, <span style="color: #f8fafc; font-weight: 500;">GICS sectors</span>, <span style="color: #f8fafc; font-weight: 500;">evidence passages</span>, and retrieval metrics (Recall@k, MRR, NDCG, MAP). If any of these terms feel unfamiliar, read the glossary first to get the most out of the analysis below.
</p>

<p style="margin: 28px 0 0 0;">
  <a href="../docs/CONTEXT.md" style="color: #5eead4; font-size: 15px; text-decoration: none; border-bottom: 1px solid #5eead4; padding-bottom: 2px;">
    &rarr;&nbsp;&nbsp;Open: docs/CONTEXT.md &mdash; Financial context and glossary
  </a>
</p>

</div>

---

# FinanceBench — Dataset exploration

**Stage 1 — Baselines · Block: FinanceBench Loader**

Exploratory notebook for the [`PatronusAI/financebench`](https://huggingface.co/datasets/PatronusAI/financebench) dataset: 150 QA pairs grounded in real SEC filings (10-K, 10-Q, 8-K) and earnings releases from publicly traded companies.

**Notebook goal**: understand the dataset's structure, coverage, and difficulty before parsing PDFs and building the indexable corpus used by downstream retrieval baselines.

---

## Setup

Load the dataset from HuggingFace. On the first run it downloads the Parquet files (~a few MB) and caches them in `~/.cache/huggingface/datasets/`. Subsequent runs are instant.

In [ ]:
# load_dataset is HuggingFace's magic function to download datasets from the Hub.
# Takes the identifier "<org>/<dataset-name>", same convention as models.
from datasets import load_dataset

# load_dataset() returns a DatasetDict — a dict-like object where:
#   - keys   = dataset splits (typically "train", "validation", "test")
#   - values = Dataset objects (HF's own class, optimized with Apache Arrow)
# The first call downloads the Parquet files and caches them in ~/.cache/huggingface/datasets/.
# Subsequent calls are instant (read from local cache).
ds = load_dataset("PatronusAI/financebench")

# FinanceBench is eval-only and ships with only 1 split. By library convention,
# any single split is named "train" even if its actual use is evaluation.
# IMPORTANT: we will NOT create train/test splits over FinanceBench — it stays
# intact as our sacred eval set. For fine-tuning we use a DIFFERENT 10-K source (Stage 3).
train = ds["train"]

# Inspect the basic structure:
#   type(ds).__name__   → "DatasetDict" (the container)
#   list(ds.keys())     → available splits
#   train.num_rows      → number of QA pairs (we expect 150)
#   train.column_names  → 15 fields (financebench_id, question, answer, evidence, etc.)
print(f"Type:    {type(ds).__name__}")
print(f"Splits:  {list(ds.keys())}")
print(f"Rows:    {train.num_rows}")
print(f"Columns: {len(train.column_names)}")

---

## Section 2 — Structure of the 150 QA pairs

We explore the dataset across **four dimensions** to understand whether it's a robust evaluation set:

1. **Documentary diversity** — how many companies, sectors, and years does it cover?
2. **Question types** — what reasoning is required (extraction vs. calculation)?
3. **Lengths** — how long are questions, answers, and evidence passages?
4. **Evidence distribution** — how many supporting passages per question?

The goal is to be able to defend the dataset in a technical interview ("is FinanceBench a robust eval set?") with data, not just intuition.

### 2.0 Dataset schema

The **15 fields** that make up each record in the dataset, in the order defined by the official schema. Useful as a quick reference when working on later sub-sections (lengths, evidence distribution, etc.).

In [ ]:
# train.column_names returns the list of columns in the order defined by the schema.
# enumerate(..., 1) starts the counter at 1 (instead of 0) so the list reads as a
# human-friendly index (1, 2, 3...) instead of a programmer index (0, 1, 2...).
# Format spec ":2d" → integer right-aligned to 2 characters (handles 1-99 without misalignment).
for i, col in enumerate(train.column_names, 1):
    print(f"{i:2d}. {col}")

#### Sample row — minimal QA structure

We print the **first record** of the dataset with all its fields. This is the **atomic unit** we'll be working with: our RAG system consumes `question`, retrieves chunks, and compares them against `evidence`.

For long fields (`question`, `evidence_text`, `justification`) we truncate the content and report the real length in parentheses, so you get a feel for the size without flooding the output.

In [ ]:
# Take the FIRST record of the dataset as a representative example.
# train[0] returns a dict {column_name: value} — the atomic unit of the dataset.
row = train[0]

# Walk through each field of the record to display it in a readable format.
# Display logic:
#   - Short strings (≤ 250 chars) → printed in full
#   - Long strings                → truncated with ellipsis AND real length reported
#   - Lists (the `evidence` case) → unpacked into their inner dicts
#   - Other types (int, None)     → printed as-is
for key, value in row.items():
    if isinstance(value, str) and len(value) > 250:
        # Report the real length in parentheses to give a sense of the size.
        print(f"{key}: ({len(value)} chars)")
        print(f"  {value[:250]}...")
    elif isinstance(value, list):
        # `evidence` is a list[dict] — show each item indexed.
        print(f"{key}: ({len(value)} items)")
        for i, item in enumerate(value):
            for k, v in item.items():
                if isinstance(v, str) and len(v) > 250:
                    print(f"  [{i}] {k}: ({len(v)} chars)")
                    print(f"      {v[:250]}...")
                else:
                    print(f"  [{i}] {k}: {v}")
    else:
        # Short fields: int (doc_period), None (domain_question_num), short strings.
        print(f"{key}: {value}")

### 2.1 Documentary diversity

**Core question**: does the dataset cover enough companies, sectors, and years for our metrics to generalize, or is it biased toward a niche?

If all 150 QA pairs were about a single company or a single sector (e.g., Tech only), our embedders could "memorize" that niche's vocabulary and produce inflated metrics that don't reflect real performance on other financial domains.

**Metrics we will measure**:

- # of distinct companies (more is better diversity)
- # of GICS sectors covered (out of 11 possible)
- Temporal range (fiscal years covered)
- Mix of document types (10-K, 10-Q, 8-K, earnings releases)

In [ ]:
# Counter (from collections) counts occurrences of each value in a list/iterable.
# Result: dict-like {value: count}, with handy methods like .most_common(n).
from collections import Counter

# train["company"] returns the entire column as a list (Arrow → Python list).
# This does NOT load the whole dataset into RAM because we extract only one column.
# Counter() turns the list into a mapping {company: # appearances}.
companies = Counter(train["company"])
sectors = Counter(train["gics_sector"])
years = Counter(train["doc_period"])
doc_types = Counter(train["doc_type"])

# === Block 1: Overview ===
# len(companies) → how many DISTINCT companies (not the total row count).
# min/max over years works because the keys are ints (fiscal years).
# dict(doc_types) converts the Counter to a regular dict for cleaner printing.
print("=" * 60)
print("DOCUMENTARY DIVERSITY — overview")
print("=" * 60)
print(f"Distinct companies: {len(companies)}")
print(f"GICS sectors:       {len(sectors)} (out of 11 possible)")
print(f"Years covered:      {min(years)}–{max(years)} ({len(years)} years)")
print(f"Document types:     {dict(doc_types)}")
print()

# === Block 2: Top 10 companies ===
# .most_common(10) returns the 10 most frequent values sorted descending.
# Format: [(value, count), ...] — we unpack in the for loop.
# This shows whether any single company dominates (which would bias the eval).
print("=" * 60)
print("TOP 10 COMPANIES BY # OF QUESTIONS")
print("=" * 60)
for company, count in companies.most_common(10):
    # f-string format spec: ":30s" = string padded to 30 chars, ":3d" = int in 3 chars.
    print(f"  {company:30s} {count:3d}")
print()

# === Block 3: Distribution by sector with ASCII bar chart ===
# .most_common() with no argument returns ALL items sorted by frequency desc.
# We compute % and draw a bar with "█" characters (1 char = 2%).
print("=" * 60)
print("DISTRIBUTION BY GICS SECTOR")
print("=" * 60)
for sector, count in sectors.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)  # each block = 2% (50% → 25 blocks max)
    print(f"  {sector:30s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 4: Temporal distribution ===
# Iterate over years in ASCENDING order to see chronological evolution.
# sorted(years.keys()) guarantees ascending order.
# We expect a strong concentration in 2022-2023 (when Patronus built the benchmark).
print("=" * 60)
print("DISTRIBUTION BY FISCAL YEAR")
print("=" * 60)
for year in sorted(years.keys()):
    count = years[year]
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {year}  {count:3d}  {bar} {pct:.1f}%")

#### Findings

**✅ Solid coverage:**

- **32 distinct companies**, no dominant outlier (max: PepsiCo with 11/150 = 7.3%) — good diversity, so our metrics aren't capturing the idiosyncrasies of a single firm.
- **9 of 11 GICS sectors** represented — covers most of the economy (only Energy and Real Estate are absent). If our embedders generalize here, they should generalize to almost any corporate financial domain.
- **10 years covered (2015–2024)** — wide temporal window.

**⚠️ Things to document as limitations:**

**1. The dataset is NOT only 10-K**, it contains a mix:

| Type | # | % | Nature |
|---|---:|---:|---|
| `10k` | 112 | 74.7% | Annual reports (audited, formal) |
| `10q` | 15 | 10.0% | Quarterly reports (unaudited) |
| `Earnings` | 14 | 9.3% | **Not SEC filings** — voluntary investor communications |
| `8k` | 9 | 6.0% | Event-driven reports |

**Technical decision (locked)**: keep the **150 mixed records** without filtering. Reason: FinanceBench is a public, standard evaluation set used in the literature; modifying it would make our results incomparable to other papers. Source heterogeneity is documented as a feature of the dataset, not as a bug.

**2. Temporal bias toward 2022–2023**: 64% of the questions target filings from those two years (2022: 40.7%, 2023: 23.3%). This reflects when Patronus AI built the benchmark. We **call this out in the final README** as a limitation: the system is being evaluated on recently published documentation.

> 📝 These decisions will be formally recorded in `docs/chunking_decisions.md` once Sub-block 6 of the current block is closed.

### 2.2 Question types

**Core question**: what kind of reasoning does each query require? Is it simple extraction ("what was the revenue?") or complex calculation ("how much did the margin grow YoY?")? This determines how challenging the dataset is.

Patronus AI labeled each question with **two independent labels** in the dataset:

- **`question_type`** — origin / style of the question
  - `metrics-generated`
  - `domain-relevant`
  - `novel-generated`
- **`question_reasoning`** — type of reasoning required to answer it
  - `Information extraction`
  - `Numerical reasoning`
  - `Logical reasoning (multi-step)`
  - `None` (`novel-generated` questions are unclassified)

**Independent** means that **one label does not determine the other** — neither is a difficulty hierarchy of the other. A `metrics-generated` question can be easy or hard; a `novel-generated` one can be trivial or complex. The difference is in their **origin**, not in their inherent complexity.

**Why did Patronus pick these categories?** Because the PDF of a 10-K has **3 distinct types of content**, and each category reflects one. The classification is NOT arbitrary — it follows the structure of the actual document.

---

#### `question_type` — origin / style of the question

##### 🟦 `metrics-generated`

**What in the PDF generates them**: a 10-K always contains **3 audited financial tables**, each one regulated by SEC GAAP rules and structurally identical across all companies:

- **Income Statement** — revenue, costs, profitability (`Net sales`, `Cost of goods sold`, `Operating income`, etc.)
- **Balance Sheet** — assets, liabilities, equity (`Total assets`, `Long-term debt`, `Stockholders' equity`, etc.)
- **Cash Flow Statement** — cash inflows/outflows (`Capital expenditures`, `Operating cash flow`, etc.)

`metrics-generated` originates specifically in these tables.

**How they're built systematically**: Patronus took a list of standard metrics from these tables and applied **rigid templates** of the form:

> *"What is the {metric} for {company} in fiscal year {year}? Answer in {unit}."*

A single template generates 50+ questions by varying the parameters (3M in 2018, AMD in 2022, Pfizer in 2023, etc.).

**Real example from 3M's PDF (2018 10-K, page 59)**:
- In the **Statement of Cash Flows** there's a line: `Purchases of property, plant and equipment ............ $(1,577)`
- Generated question: *"What is the FY2018 capital expenditure amount in USD millions for 3M?"*
- Answer literal in the PDF: **$1,577M**

**Why Patronus created it**: it guarantees **systematic coverage** of the metrics most used in financial analysis, and lets you **compare results across companies** with the same question varying only the subject.

##### 🟨 `domain-relevant`

**What in the PDF generates them**: a 10-K has a **predictable structure** regulated by the SEC. Beyond the audited financial tables, it has **mandatory free-prose narrative sections**. The ones that generate `domain-relevant` are:

- **Item 1** — Business (operations, products, markets, competition)
- **Item 1A** — Risk Factors (specific material risks the company identifies)
- **Item 7** — MD&A (Management's Discussion and Analysis — management's narrative on performance)

*(The complete anatomy of a 10-K — all Items and Parts — is documented in `docs/CONTEXT.md` §3 — Filings.)*

These sections are **predictable in their existence** (every 10-K has them) but **variable in their content** (each company describes its own risks).

**How they're built**: humans wrote questions with **clear focus on these sections** but without sticking to a template. The questions are thematically predictable ("what risks does it identify?", "what is the growth strategy?") but the format is free.

**Real example from 3M's PDF (2018 10-K, Item 1A)**:
- The PDF has several paragraphs describing risks: PFAS contamination, pending litigation, supply chain disruption, etc.
- Question: *"What are the main risks 3M identifies in its 2018 annual report?"*
- Answer: **requires READING and SYNTHESIZING several prose paragraphs** — there's no single value to extract.

**Why Patronus created it**: questions about financial statements are easy to template, but real analysts also ask about **the qualitative context** (strategy, risks, governance). This category covers that ground.

##### 🟥 `novel-generated`

**What in the PDF generates them**: instead of staying within **a single table** (like `metrics-generated`) or **a single narrative section** (like `domain-relevant`), novel questions **combine signals from multiple parts** of the document. For example:

- A **metric** from the financial statements (Income Statement / Balance Sheet / Cash Flow Statement)
- A **narrative** from the qualitative items (Item 1 Business / Item 7 MD&A)
- A **risk** from Item 1A

**How they're built**: humans wrote these questions **completely open**, simulating how a junior analyst asks a colleague or how a curious investor explores a 10-K without a predefined agenda.

**Real example from Pfizer's PDF (2022 10-K)**:
- The PDF has: (a) R&D spend in the Income Statement, (b) description of the oncology pipeline in Item 1 Business, (c) competitive risks in Item 1A.
- Question: *"How is Pfizer's R&D pipeline positioning the company in oncology?"*
- Answer: **requires crossing 3 sections of the PDF** — the financial tables (how much is invested), the Business narrative (what is being built), and the Risk Factors (what competitive threats exist).

**Why Patronus created it**: it's the **stress test** of the benchmark. If your RAG system does well on `metrics-generated` (templated, predictable questions) but **collapses on `novel-generated`** (free-form questions), that reveals the system **memorizes the prompt format** instead of understanding semantics.

---

#### `question_reasoning` — required reasoning

**What does this label classify?** How much cognitive work the system has to do **after** finding the correct chunk. The 3 categories reflect **where the answer lives in the PDF**: literal in the text, computed from PDF data, or derived by combining multiple pieces.

##### 🔵 `Information extraction`

**How it looks in the PDF**: the answer is **literal in a table cell or in a direct mention in prose**. The system just needs to find the correct chunk and return the text.

**Real example**:
- 3M's PDF (2018 10-K, Income Statement): line `Net sales ............... $32,765`
- Question: *"What was 3M's revenue in FY2018?"*
- Direct answer: **$32,765M** — already literal in the PDF, no calculation needed.

##### 🟢 `Numerical reasoning`

**How it looks in the PDF**: the answer is **NOT literal** in the PDF — the document contains the **inputs** (2 or more values), but the result has to be **computed**.

**Real example**:
- 3M's PDF: `Net sales 2017 = $31,657` and `Net sales 2018 = $32,765`
- Question: *"What was 3M's revenue YoY growth from 2017 to 2018?"*
- Answer: **NOT in the PDF** — must be computed: `(32,765 - 31,657) / 31,657 = 3.5%`.

The RAG system **must retrieve the chunk** with both numbers (ideally the full table showing the year-over-year comparison). The calculation is done by the downstream LLM.

##### 🟣 `Logical reasoning (multi-step)`

**How it looks in the PDF**: the answer requires **combining multiple pieces of information**, possibly from **different sections of the PDF** (e.g., Cash Flow Statement + Income Statement). You have to extract data, compute intermediate ratios, compare results, and emit a judgment.

**Real example**:
- Question: *"Did 3M's CapEx grow faster than its revenue between 2017 and 2018?"*
- Inputs in the PDF (4 numbers, 2 different sections):
  - **Income Statement**: Revenue 2017 = $31,657M, Revenue 2018 = $32,765M
  - **Cash Flow Statement**: CapEx 2017 = $1,373M, CapEx 2018 = $1,577M
- Process:
  1. Extract the 4 numbers
  2. Compute CapEx growth = `(1,577 - 1,373) / 1,373 = 14.9%`
  3. Compute Revenue growth = `(32,765 - 31,657) / 31,657 = 3.5%`
  4. Compare: 14.9% > 3.5% → **Yes, CapEx grew faster**

The RAG system **must retrieve chunks from 2 different sections** of the 10-K. This connects directly with what we'll see in Sub-block 2.4: **~23% of the dataset has 2-3 distinct evidences** because multi-step questions require information from various places in the PDF.

> 💡 **Summary table + glossary of categories**: see [`docs/CONTEXT.md` § How Patronus categorized the questions](../docs/CONTEXT.md#how-patronus-categorized-the-questions).

In [ ]:
# Counter over question_type — distribution of the 3 general categories.
q_types = Counter(train["question_type"])

# question_reasoning can be None in ~33% of the rows (all novel-generated).
# Replace None with the string "(None)" in a generator expression so Counter
# doesn't choke when iterating, and the output clearly shows unclassified rows.
q_reasoning = Counter(
    str(x) if x is not None else "(None)"
    for x in train["question_reasoning"]
)

# === Block 1: question_type ===
print("=" * 65)
print("QUESTION_TYPE — general question category")
print("=" * 65)
for qt, count in q_types.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {qt:30s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 2: question_reasoning ===
print("=" * 65)
print("QUESTION_REASONING — required reasoning type")
print("=" * 65)
for qr, count in q_reasoning.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {qr:35s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 3: Cross-tab question_type × question_reasoning ===
# defaultdict(lambda: defaultdict(int)) creates a NESTED dict that auto-initializes
# any missing key with 0. Useful for accumulating counts without existence checks.
from collections import defaultdict
cross = defaultdict(lambda: defaultdict(int))
for r in train:
    qr_key = r["question_reasoning"] if r["question_reasoning"] is not None else "(None)"
    cross[r["question_type"]][qr_key] += 1

print("=" * 65)
print("CROSS-TAB question_type × question_reasoning")
print("=" * 65)
for qt in sorted(cross.keys()):
    print(f"\n[{qt}]")
    # Sort by count desc within each category → most common reasoning shows first.
    for qr, c in sorted(cross[qt].items(), key=lambda x: -x[1]):
        print(f"  {qr:50s} {c:3d}")

#### Findings

**1. `question_type` is PERFECTLY balanced** (50/50/50):

| Category | # | Nature |
|---|---:|---|
| `metrics-generated` | 50 | Templated questions about metrics (CapEx, revenue, margin) |
| `domain-relevant` | 50 | Broader domain questions (strategy, risks, governance) |
| `novel-generated` | 50 | Novel questions, no template |

Patronus AI designed the dataset with equal quotas — good news for our metrics, no bias toward any single type.

**2. `question_reasoning` shows real heterogeneity**:

- ~29% **Numerical reasoning** (calculations)
- ~21% **Information extraction** (direct lookup)
- ~17% **Logical hybrids** (combinations of Logical + Numerical multi-step)
- 33% **`None`** (all `novel-generated`, unclassified)

**3. Cross-tab key insight**:

- `metrics-generated` → 72% Numerical + 28% Information extraction (templated questions are calculations or lookups)
- `domain-relevant` → broader diversity, with many multi-step hybrid cases (the most complex questions)
- `novel-generated` → all with `reasoning = None` (Patronus left them unclassified)

**Project implication**: the ~29% Numerical reasoning requires that the RAG system not only retrieves the correct chunk, but that the downstream LLM performs the calculation. Since Stage 1-2 measures **retrieval only**, we focus on whether the retrieved chunk contains the necessary numbers — the calculation is the generation layer's responsibility (out of scope here, addressed in Eslabón 2).

> 📝 Defensive against `None`: any code iterating `question_reasoning` must handle None values (33% of the dataset).

#### Tagging in action — seeing how it applies to real questions

So far we saw the **aggregate counts** (50/50/50, etc.) and the **hypothetical examples** from the PDF. Now we close the pedagogical loop: we take **3 real questions from the dataset** (one per `question_type`) and display **all their tags + metadata + an evidence fragment**.

This shows how Patronus applied the categorization to concrete questions, and how each category has a distinct "feel" when you read it:

- **`metrics-generated`** feels like a standardized exam (predictable structure, exact numerical value)
- **`domain-relevant`** feels like an analytical conversation (clear focus, free format)
- **`novel-generated`** feels like a natural question (no pattern, exploratory)

> 💡 **Tip**: change the `random.seed(42)` to another number in the cell below to explore other examples from the dataset and see the real diversity of each category.

In [ ]:
import random

# We fix the seed for reproducibility: every time you run this cell, the same
# 3 examples are chosen. Change the seed to explore other dataset cases.
random.seed(42)


def show_tagged_example(category: str, max_evidence_chars: int = 250) -> None:
    """Show ONE random example of the indicated question_type category,
    displaying all its tags + metadata + a fragment of the evidence.

    Educational: connects the theory (question type + reasoning) with the real dataset.
    """
    # Filter the dataset by question_type and pick one at random.
    filtered = [r for r in train if r["question_type"] == category]
    example = random.choice(filtered)

    print("=" * 80)
    print(f"📌 EXAMPLE: question_type = {category}")
    print("=" * 80)

    print(f"\n💬 Question:\n  {example['question']}\n")

    # The 2 main tags we've been analyzing above.
    print("🏷️  Tags:")
    print(f"  question_type:       {example['question_type']}")
    print(f"  question_reasoning:  {example['question_reasoning']}\n")

    # Source document metadata — useful to locate the evidence in the actual PDF.
    print("📄 Source document:")
    print(f"  company:             {example['company']}")
    print(f"  doc_name:            {example['doc_name']}")
    print(f"  doc_type / period:   {example['doc_type']} / {example['doc_period']}")
    print(f"  gics_sector:         {example['gics_sector']}\n")

    # Expected answer. For Information extraction it's a literal value;
    # for Numerical reasoning it's usually a computed number; for Logical multi-step
    # it can be a yes/no judgment or a textual conclusion.
    print(f"✅ Answer:\n  {example['answer']}\n")

    # Evidence: PDF passage(s) that justify the answer. Truncated for readability.
    # If there's more than 1 item, these are the multi-evidence QAs (~23% of dataset).
    print(f"📑 Evidence ({len(example['evidence'])} item(s)):")
    for i, item in enumerate(example["evidence"]):
        page = item.get("evidence_page_num", "N/A")
        text = item.get("evidence_text", "")
        truncated = text[:max_evidence_chars] + ("..." if len(text) > max_evidence_chars else "")
        print(f"  [{i}] page {page}:")
        # Indent the evidence text so it stands out visually.
        for line in truncated.split("\n")[:6]:
            print(f"      {line}")
    print()


# Call the function for the 3 question_type categories.
# Each call picks a different random example (reproducible thanks to the seed).
for cat in ["metrics-generated", "domain-relevant", "novel-generated"]:
    show_tagged_example(cat)
    print()

#### Important clarification about `question_reasoning = None`

A common intuition when looking at the data: *"`novel-generated` questions have `reasoning = None` because they require more complex post-processing from the LLM."*

**That's NOT accurate.** `question_reasoning = None` means **Patronus did not classify them** by reasoning type — NOT that they necessarily require more processing. A `novel-generated` question can be simple extraction or complex numerical; Patronus left them free-form because their diversity makes them hard to bucket.

**But the broader intuition IS correct**: ALL questions involving numerical or logical reasoning (~46% of the dataset) need post-processing from the downstream LLM. That's **not retrieval** — it's generation, living in a different phase of the RAG pipeline.

> 💡 **The 3 RAG pipeline phases (retrieval / reranking / generation) and what Stage 1-2 covers vs Eslabón 2**: see [`docs/CONTEXT.md` §10 — The RAG pipeline](../docs/CONTEXT.md#10-the-rag-pipeline--retrieval-reranking-generation).

### 2.3 Lengths (questions, answers, evidence)

**Core question**: how long are the dataset's texts? This defines two critical things for the RAG system:

1. **Embedder token budget** — modern embedders have a limit (typically 512 tokens). If our chunks or queries exceed it, we must truncate (losing information) or switch models.
2. **Appropriate chunking strategy** — if `evidence` passages are very long, we need bigger chunks OR retrieval that returns multiple chunks to cover the full evidence.

**What we measure**:

- Distribution (min, p25, median, p75, p95, max, mean) of each field in chars
- Token estimation (rule of thumb: 1 token ≈ 4 chars in English)
- Distribution of # of items per evidence (1, 2, or 3 passages per question)

We use `statistics.quantiles` from stdlib (no pandas/numpy) — the dataset is small, no need for heavy artillery.

> 💡 **Quick refresher**: if the terms `min`, `p25`, `median`, `p75`, `p95`, `max`, `mean` aren't familiar, read [`docs/CONTEXT.md` § 9 — Descriptive statistics primer](../docs/CONTEXT.md#9-descriptive-statistics-primer-min-mean-median-percentiles) before continuing.

In [ ]:
import statistics as stats

# Helper to print the distribution of a list of numeric values in a compact format.
# Reports: count, min, p25, median, p75, p95, max, mean.
#   stats.quantiles(values, n=4)  → quartiles: [p25, p50, p75]
#   stats.quantiles(values, n=20) → vigintiles: index 18 = p95 (19/20)
def describe(name, values):
    p25 = int(stats.quantiles(values, n=4)[0])
    p75 = int(stats.quantiles(values, n=4)[2])
    p95 = int(stats.quantiles(values, n=20)[18])
    print(f"{name:25s}  n={len(values):3d}  min={min(values):5d}  p25={p25:5d}  "
          f"median={int(stats.median(values)):5d}  p75={p75:5d}  p95={p95:5d}  "
          f"max={max(values):5d}  mean={int(stats.mean(values)):5d}")


# === Compute lengths in CHARS ===
# question, answer: simple strings, len() directly.
question_lens = [len(r["question"]) for r in train]
answer_lens = [len(r["answer"]) for r in train]

# justification can be None (~25% of dataset). We treat None as length 0.
# This keeps the count at 150 rows and honestly reflects missing justifications.
justification_lens = [len(r["justification"]) if r["justification"] else 0 for r in train]

# evidence is list[dict]. For each QA, we sum the chars of ALL evidence_text items.
# This reflects the "total evidence load" the system would need to retrieve.
evidence_total_lens = [
    sum(len(item["evidence_text"]) for item in r["evidence"])
    for r in train
]

# === Block 1: Distribution in chars ===
print("=" * 110)
print("LENGTHS (in chars) — distribution")
print("=" * 110)
describe("question", question_lens)
describe("answer", answer_lens)
describe("justification", justification_lens)
describe("evidence (sum per QA)", evidence_total_lens)
print()

# === Block 2: Token estimation ===
# Rule of thumb for English: ~4 chars per token on average.
# (Spanish: ~3 chars/token. Source code: ~2 chars/token.)
# For exact tokens, you'd pass the text through the target model's tokenizer.
print("=" * 110)
print("TOKEN estimation (rule of thumb: 1 token ≈ 4 chars in English)")
print("=" * 110)
print(f"  question      median ~{int(stats.median(question_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(question_lens, n=20)[18]/4):3d} tokens")
print(f"  answer        median ~{int(stats.median(answer_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(answer_lens, n=20)[18]/4):3d} tokens")
print(f"  evidence/QA   median ~{int(stats.median(evidence_total_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(evidence_total_lens, n=20)[18]/4):3d} tokens")
print()

# === Block 3: # of evidence items per question ===
# How many evidence pieces each QA has. Schema says 1-3 items.
evidence_counts = [len(r["evidence"]) for r in train]
counts_dist = Counter(evidence_counts)

print("=" * 110)
print("# of evidence items per question")
print("=" * 110)
for n in sorted(counts_dist.keys()):
    c = counts_dist[n]
    pct = c / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {n} items  {c:3d}  {bar} {pct:.1f}%")

#### Findings

**1. Questions and answers are small, evidence is the diva of the show**:

| Field | median | p95 | max | Estimated tokens (median) |
|---|---:|---:|---:|---|
| `question` | 137 chars | 356 | 592 | ~34 tokens |
| `answer` | 50 chars | 285 | 609 | ~12 tokens |
| `justification` | 100 chars | 488 | 703 | ~25 tokens |
| **`evidence` (sum per QA)** | **1,450 chars** | **4,194** | **12,123** | **~362 tokens** |

- Questions and answers fit comfortably in any embedder (max 512 tokens).
- `justification` may be empty for ~25% of the dataset (`min=0`, `p25=0`).
- `evidence` has a **very long tail**: the outlier reaches 12,123 chars (~3,030 tokens).

**2. Critical chunking implication**:

If we chunk at **512 tokens (~2,048 chars)** — the standard convention for BERT-family embedders:

| Evidence percentile | Fits in 1 chunk? |
|---|---|
| Median (1,450 chars / ~362 tokens) | ✅ Plenty of room |
| p75 (2,267 chars / ~566 tokens) | ⚠️ Just barely overflows → needs 2 chunks |
| p95 (4,194 chars / ~1,048 tokens) | ❌ Needs 2-3 chunks |
| Max (12,123 chars / ~3,030 tokens) | 🚨 Outlier — needs 6+ chunks |

For ~25-30% of the dataset, the full evidence does NOT fit in 1 chunk. **This validates our choice to measure Recall@k with k=5,10** (not just k=1) and to include **MAP** as a metric — both reward retrieving multiple chunks when the evidence spans across them.

**3. # of evidence items per question**:

- **76.7%** has **1 item** (1 PDF passage justifies the answer)
- **20.7%** has **2 items**
- **2.7%** has **3 items**

**~23% of the dataset has multiple distinct evidences**. These are the cases where MRR misleads (it only looks at the first hit) and **MAP is honest** (it looks at all hits).

> 📝 These measurements will inform `docs/chunking_decisions.md` (sub-block 6) when we justify the chosen chunk size.

### 2.4 Evidence distribution

**Core question**: when the RAG system performs retrieval, **how deep into the 10-K does it have to look?** Do answers live at the beginning (executive summary), in the middle (MD&A + financial statements), or scattered throughout?

Three dimensions we measure:

1. **Distribution of `evidence_page_num`** — which pages do evidences live on?
2. **Multi-evidence locality** — when a QA has 2-3 evidences, do they come from nearby or far-apart pages?
3. **Ratio `evidence_text` vs `evidence_text_full_page`** — is the evidence almost the whole page or a small fragment?

**Why it matters**:

- **If evidences live in a bounded range** (e.g., pages 0-100), there's no point indexing the whole document — we can truncate initial pages (cover, TOC) and final ones (signatures, exhibits) → token and compute savings.
- **If multi-evidences are far apart**, we need diversity-aware retrieval (MMR / reranking). If they're nearby, vanilla top-k with cosine similarity is enough.
- **If `evidence_text` is almost the whole page**, full-page chunking works. If it's a small fragment, we need finer-grained chunks.

In [ ]:
# === Block 1: Distribution of evidence_page_num ===
# Collect ALL page numbers from ALL evidence items (we don't aggregate per QA).
# Defensive: some evidence items may have page_num = None — skip those.
all_pages = []
for r in train:
    for item in r["evidence"]:
        page = item.get("evidence_page_num")
        if page is not None:
            all_pages.append(page)

print("=" * 75)
print(f"EVIDENCE_PAGE_NUM (n={len(all_pages)} total items across 150 QAs)")
print("=" * 75)
print(f"min={min(all_pages):3d}  p25={int(stats.quantiles(all_pages, n=4)[0]):3d}  "
      f"median={int(stats.median(all_pages)):3d}  p75={int(stats.quantiles(all_pages, n=4)[2]):3d}  "
      f"p95={int(stats.quantiles(all_pages, n=20)[18]):3d}  max={max(all_pages):3d}  "
      f"mean={int(stats.mean(all_pages)):3d}")
print()

# Manual histogram by page ranges that map to typical 10-K sections.
# The ranges are NOT uniform — they're chosen to separate natural 10-K items
# (Item 1 ~pp.0-25, MD&A ~pp.26-50, Financial Statements ~pp.51-75, etc.).
ranges = [(0, 25), (26, 50), (51, 75), (76, 100), (101, 150), (151, 250), (251, 500)]
print("Distribution by page range:")
for lo, hi in ranges:
    count = sum(1 for p in all_pages if lo <= p <= hi)
    pct = count / len(all_pages) * 100
    bar = "█" * int(pct / 2)
    print(f"  pp.{lo:3d}-{hi:3d}  {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 2: Multi-evidence locality ===
# For QAs with 2+ evidences, compute the "distance" between pages: max(pp) - min(pp).
#   distance = 0 → all evidences on the same page
#   distance = 5 → evidences within a 5-page window
# Tells whether multi-evidences are "clustered" (same section) or "scattered".
multi_qa_distances = []
for r in train:
    if len(r["evidence"]) >= 2:
        pages = [
            item["evidence_page_num"]
            for item in r["evidence"]
            if item.get("evidence_page_num") is not None
        ]
        if len(pages) >= 2:
            distance = max(pages) - min(pages)
            multi_qa_distances.append(distance)

print("=" * 75)
print(f"MULTI-EVIDENCE LOCALITY (n={len(multi_qa_distances)} QAs with 2+ evidences)")
print("=" * 75)
print("Distance between evidences (max page - min page):")
print(f"  min={min(multi_qa_distances):3d}  p25={int(stats.quantiles(multi_qa_distances, n=4)[0]):3d}  "
      f"median={int(stats.median(multi_qa_distances)):3d}  p75={int(stats.quantiles(multi_qa_distances, n=4)[2]):3d}  "
      f"max={max(multi_qa_distances):3d}  mean={int(stats.mean(multi_qa_distances)):3d}")
print()

dist_buckets = [
    (0, 0, "same page"),
    (1, 5, "near (1-5 pp)"),
    (6, 25, "far (6-25 pp)"),
    (26, 100, "very far (26-100 pp)"),
    (101, 1000, "extreme (>100 pp)"),
]
print("Distance distribution:")
for lo, hi, label in dist_buckets:
    count = sum(1 for d in multi_qa_distances if lo <= d <= hi)
    pct = count / len(multi_qa_distances) * 100
    bar = "█" * int(pct)  # 1:1 scale (each block = 1%) — the tail is small, this reads better
    print(f"  {label:25s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 3: Ratio evidence_text vs evidence_text_full_page ===
# Each evidence item carries 2 strings:
#   evidence_text           → the RELEVANT passage (human-curated)
#   evidence_text_full_page → the full page where the passage lives
# The ratio tells what fraction of the page is the "real" evidence:
#   ratio close to 1.0 → the evidence IS almost the whole page (page = table)
#   ratio close to 0.0 → the evidence is a small fragment of a dense page
ratios = []
for r in train:
    for item in r["evidence"]:
        et = len(item.get("evidence_text", ""))
        full = len(item.get("evidence_text_full_page", ""))
        if full > 0:
            ratios.append(et / full)

print("=" * 75)
print(f"RATIO evidence_text / evidence_text_full_page (n={len(ratios)} items)")
print("=" * 75)
print(f"min={min(ratios):.2f}  p25={stats.quantiles(ratios, n=4)[0]:.2f}  "
      f"median={stats.median(ratios):.2f}  p75={stats.quantiles(ratios, n=4)[2]:.2f}  "
      f"max={max(ratios):.2f}  mean={stats.mean(ratios):.2f}")
print()

ratio_buckets = [
    (0.0, 0.25, "small fragment (<25%)"),
    (0.25, 0.5, "medium fragment (25-50%)"),
    (0.5, 0.9, "majority (50-90%)"),
    (0.9, 1.01, "almost full page (>90%)"),
]
print("Ratio distribution:")
for lo, hi, label in ratio_buckets:
    count = sum(1 for r in ratios if lo <= r < hi)
    pct = count / len(ratios) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:30s} {count:3d}  {bar} {pct:.1f}%")

#### Findings

**1. ~85% of evidences live between pages 0-75** of the 10-K:

| Range | % | Maps to typical 10-K section |
|---|---:|---|
| pp. 0-25 | 27.5% | Item 1 (Business) + Item 1A (Risk Factors) |
| pp. 26-50 | 19.0% | Item 7 (MD&A) opening |
| **pp. 51-75** ⭐ | **38.1%** | **Item 8 (Financial Statements)** — the heart of the 10-K |
| pp. 76-100 | 7.9% | Notes to Financial Statements |
| pp. 101+ | 7.5% | Appendices and exhibits |

`mean ≈ median` (both = 51) → fairly symmetric distribution, no tails distorting the center.

**2. Multi-evidence locality: 91% are within ≤5 pages of each other**:

When a QA has 2-3 evidences, they almost always come from the **same section** of the 10-K (e.g., both from the Cash Flow Statement, or both from the Balance Sheet).

**Technical implication**: we do NOT need **MMR** (Maximal Marginal Relevance) or diverse reranking for the typical case — the relevant chunks are clustered, and top-k with cosine similarity captures them well. Only the ~9% of outliers (multi-evidence cross-section) would require something more sophisticated.

**3. Bimodal distribution of evidence/page ratio**:

- **46% of evidences cover almost the whole page** (>90%) — typical when the page is a complete financial table (Income Statement, Balance Sheet).
- **29% are small fragments** (<25%) — typical when the page has multiple topics and the evidence is a specific paragraph.

**Critical trade-off the chunking-strategy comparison will measure**:

| Strategy | How it does |
|---|---|
| Chunk = full page | ✅ Perfect for 46% (tables) · ❌ Dilutes small fragments |
| Chunk = 512 tokens | ✅ Captures small fragments · ❌ May split large tables |
| **Semantic chunking** | ✅ Respects semantic boundaries |
| **Late chunking** | ✅ Embeds the whole doc first, decides chunks afterward |

**This is why the project uses 4 strategies** instead of committing to one — each has strengths in different parts of the distribution.

> 📝 These measurements will be documented in `docs/chunking_decisions.md` (sub-block 6) when we justify which strategy wins in each ratio quartile.

---

## 🧭 Sub-block 2 wrap-up — what we learned about the dataset

After exploring the 5 sub-sections, we can tell the story of FinanceBench as a single coherent piece:

### 1. The dataset is a serious benchmark, not a toy example

The 150 questions are not "financial trivia". They're real queries a junior analyst would ask at an investment bank or fund:

- **32 distinct companies** across **9 economic sectors** (everything except Energy and Real Estate)
- **10 years of coverage** (2015-2024), though biased toward 2022-2023
- Balanced mix of three question types: **templated** (calculate ratios), **broad-domain** (strategy, risks), and **novel** (no fixed pattern)

> **Translation**: if a RAG system works here, there's a good chance it'll work on any real-world corporate financial domain.

### 2. The system doesn't just have to "find" — it has to understand

The `question_reasoning` field reveals three difficulty levels:

- **~21% direct lookup**: *"what was the revenue?"* — the system just needs to find the number.
- **~29% require calculation**: *"how much did it grow YoY?"* — the system finds two numbers and subtracts them.
- **~17% multi-step reasoning**: *"did CapEx grow faster than revenue?"* — multiple extractions, ratios, comparison.

> **Translation**: our project focuses only on the first half of the job (finding the right chunk). The calculation and reasoning are the downstream LLM's responsibility, which lives in Eslabón 2 of the roadmap.

### 3. There's a clear geographic pattern within the 10-K

The `evidence` items aren't randomly scattered — they live mostly in a specific zone of the document:

- **85% in the first 75 pages** of the 10-K
- Strong concentration in **pages 51-75**: the heart of the document (financial statements: Income, Balance Sheet, Cash Flow)
- When there are multiple evidences per question (~23% of dataset), **91% are within ≤5 pages of each other** — same section

> **Translation**: there's no need to read the entire 10-K to answer. Most answers live in a predictable zone. This justifies future optimizations (e.g., indexing only pages 0-150) and allows simple retrieval (vanilla top-k) instead of complex diversity algorithms.

### 4. The dataset is honest about its own complexity

The `evidence_text_full_page` field reveals a critical trade-off:

- **46% of evidences are a full page** (typically a financial table)
- **29% are small fragments** within dense pages
- The rest fall in the middle

> **Translation**: there's NO single chunking strategy that wins all cases. If we chunk to full page, we fail with fragments. If we chunk small, we split tables. **That's why the project compares 4 strategies** — each one optimized for a different dataset profile.

---

### What technical decisions this data justifies

| Technical decision | Justified by Sub-block 2 |
|---|---|
| Report **4 metrics** (Recall@k, MRR, NDCG, MAP) | Multi-evidence exists (~23%), MRR misleads, MAP captures everything |
| Report multiple `k` values (k=1, 3, 5, 10) | Evidence may not fit in 1 chunk (long-tail length distribution) |
| Compare **4 chunking strategies** | Bimodal evidence/page ratio distribution (no one-size-fits-all) |
| Keep the dataset **mixed** (don't filter to 10-K only) | Filtering would make us incomparable with academic literature |
| **Don't** invest in MMR / diverse retrieval for the baseline | Multi-evidences are nearby in 91% of cases |

This closes **Sub-block 2** of the FinanceBench Loader block.

## 3. Parsing PDFs with `pdfplumber`

> **Goal of §3**: convert the 84 10-K PDFs (downloaded in sub-block 3) into a structured indexable corpus that preserves text + table structure for downstream chunking and embedding.

The full conceptual breakdown lives in [`docs/CONTEXT.md`](../docs/CONTEXT.md) and the Notion descriptive memory of this block. Here we **materialize the 5 technical decisions** with executable demos against a real 3M 10-K page.


### 3.0 The fundamental problem — PDF doesn't contain text

PDF descends from PostScript: it describes how to *print* a page, not how to *represent* content. When you see *"Net sales 32,765"* on screen, the PDF stores **glyphs positioned at (x, y) coordinates** — there's no native concept of "paragraph", "table", "header", or "cell". Every PDF parsing library has to reconstruct structure from those raw glyph coordinates.

Let's see this directly: open the 3M 2018 10-K Consolidated Statement of Income and inspect the raw `page.chars`.


In [1]:
# === Block 1: Open the PDF and inspect raw glyphs (chars) ===
# pdfplumber.open() returns a context-managed PDF object. Always use 'with'
# so the underlying file handle closes cleanly even if parsing fails mid-page.
import pdfplumber

# Path is relative to the notebook's location (notebooks/), hence the '../'.
PDF_PATH = "../data/raw/pdfs/3M_2018_10K.pdf"

# Page index 55 (0-indexed) is the Consolidated Statement of Income in 3M's
# 2018 10-K — verified in pre-flight by searching for "Cost of sales" + "32,765".
# Hardcoded for the demo; production code (scripts/parse_pdfs.py) iterates ALL pages.
INCOME_STMT_IDX = 55

with pdfplumber.open(PDF_PATH) as pdf:
    # len(pdf.pages) reads the page count from the PDF header — O(1), no parsing.
    print(f"Total pages in 3M_2018_10K.pdf: {len(pdf.pages)}")

    # Indexing pdf.pages[i] is lazy: the page is only parsed when its attributes
    # (chars, extract_text, extract_tables) are accessed — not on indexing itself.
    page = pdf.pages[INCOME_STMT_IDX]

    # page.page_number is 1-indexed (PDF convention). page.chars is the canonical
    # low-level access: every glyph (rendered character) on the page with full
    # positional + font metadata. This is THE evidence that PDF stores positioned
    # glyphs, not text — the foundational concept of §3.0.
    print(f"\nInspecting page_number={page.page_number}")
    print(f"Total raw glyphs (chars) on this page: {len(page.chars)}")

    # Each ch is a dict with ~15 keys; we print the most informative 4:
    #   text:     the actual character (e.g., 'N', 'e', '3')
    #   x0, top:  pixel coordinates on the page (origin = top-left)
    #   fontname: PDF-internal font ID (often gibberish like 'BCDEEE+ArialMT')
    #   size:     font size in points (≈1.33 pixels at 96 DPI)
    print(f"\nFirst 8 glyphs — note each has its own (x, y) coordinate, font, and size:")
    for ch in page.chars[:8]:
        print(
            f"  '{ch['text']}' at (x0={ch['x0']:6.1f}, top={ch['top']:6.1f})  "
            f"| font={ch['fontname']:25s}  | size={ch['size']:.1f}"
        )


Total pages in 3M_2018_10K.pdf: 160

Inspecting page_number=56
Total raw glyphs (chars) on this page: 1346

First 8 glyphs — note each has its own (x, y) coordinate, font, and size:
  'T' at (x0=  56.7, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'a' at (x0=  61.5, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'b' at (x0=  64.9, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'l' at (x0=  68.8, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'e' at (x0=  71.0, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  ' ' at (x0=  74.4, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'o' at (x0=  76.4, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8
  'f' at (x0=  80.3, top=  20.6)  | font=QIOBAA+TimesNewRomanPSMT   | size=7.8


### 3.1 Why tables are the worst case — the flattening problem

Tables are inherently **2D** (rows × columns). When a naive parser extracts them reading top-to-bottom, left-to-right, it **flattens** the structure into a linear string:

> *"2018 2017 2016 Net sales 32,765 31,657 30,109 Operating income 7,207 6,920 6,494"*

The row ↔ column association — which lived in the spatial coordinates — is lost in the flattening. Now the downstream system has no way to know that `32,765` corresponds to **Net sales in 2018**.

`pdfplumber` solves this by exposing two distinct APIs:

- `extract_text()` → returns the flattened narrative (good for prose like Item 1A, MD&A).
- `extract_tables()` → returns each detected table as a **2D matrix** preserving rows and columns.

Let's see both APIs operating on the Income Statement page — and surface a real gotcha along the way.


In [2]:
# === Block 1: extract_text() — narrative + flattened tables ===
# We re-open the PDF (pdfplumber doesn't cache parsed pages across 'with' blocks).
# In production, you'd parse all pages of a PDF inside ONE 'with' block to avoid
# re-opening overhead — see scripts/parse_pdfs.py.
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[INCOME_STMT_IDX]

    # extract_text() returns a single string with the entire page's text, in
    # reading order (top-to-bottom, left-to-right). It's good for narrative
    # sections (Item 1A, MD&A, Notes prose) but FLATTENS any table on the page —
    # this is the failure mode we're surfacing.
    text = page.extract_text()
    print("=== extract_text() — first 400 chars ===")
    print(text[:400])
    print(f"\n[total length: {len(text)} chars]")

    # === Block 2: extract_tables() with DEFAULT settings — surfaces the gotcha ===
    # extract_tables() runs pdfplumber's heuristic table detector and returns each
    # detected table as List[List[str]]. The DEFAULT strategy uses ruling lines
    # (vertical_strategy='lines', horizontal_strategy='lines') — which is exactly
    # WHY the Income Statement gets fragmented: each row has its own horizontal
    # separator line, so the heuristic treats each row as a separate "table".
    print("\n" + "=" * 60)
    print("=== extract_tables() with DEFAULT settings ===")
    tables_default = page.extract_tables()
    print(f"Tables detected: {len(tables_default)}")
    print("Note: the default heuristic FRAGMENTS this single Income Statement")
    print("into ~12 sub-tables because of horizontal separator lines between rows.")
    print("This IS the gotcha #2 from the Notion memory: false-positive tables.")

    # === Block 3: extract_tables() with text-based strategy — the fix ===
    # 'text' strategy uses character clustering instead of ruling lines: it groups
    # nearby glyphs into rows/columns based on x/y proximity. This is robust to
    # tables-with-separators, tables-without-borders, and most edge cases in 10-K.
    # It becomes the project-default table_settings (see scripts/parse_pdfs.py).
    print("\n" + "=" * 60)
    print("=== extract_tables() with text-based strategy (the fix) ===")
    settings = {"vertical_strategy": "text", "horizontal_strategy": "text"}
    tables_clean = page.extract_tables(settings)
    print(f"Tables detected: {len(tables_clean)}")
    # tables_clean[0] is the matrix; .[0] is the first row (header).
    # Reading the shape confirms: 33 rows × 10 cols = the full Income Statement.
    print(f"Shape: {len(tables_clean[0])} rows × {len(tables_clean[0][0])} cols")
    print("\nFirst 6 rows of the cleanly-extracted Income Statement:")
    for row in tables_clean[0][:6]:
        print(f"  {row}")


=== extract_text() — first 400 chars ===
Table of Contents
3M Company and Subsidiaries
Consolidated Statement of Incom e
Years ended December 31
(Millions, except per share amounts) 2018 2017 2016
Net sales $ 32,765 $ 31,657 $ 30,109
Operating expenses
Cost of sales 16,682 16,055 15,118
Selling, general and administrative expenses 7,602 6,626 6,311
Research, development and related expenses 1,821 1,870 1,764
Gain on sale of businesses (5

[total length: 1240 chars]

=== extract_tables() with DEFAULT settings ===
Tables detected: 12
Note: the default heuristic FRAGMENTS this single Income Statement
into ~12 sub-tables because of horizontal separator lines between rows.
This IS the gotcha #2 from the Notion memory: false-positive tables.

=== extract_tables() with text-based strategy (the fix) ===
Tables detected: 1
Shape: 33 rows × 10 cols

First 6 rows of the cleanly-extracted Income Statement:
  ['Table of Contents', '', '', '', '', '', '', '', '', '']
  ['', '', '', '', '', '', '', '

### 3.2 Header-repetition strategy for large tables

Even with `pdfplumber` extracting the matrix correctly, a problem remains in the chunking layer: **what do we do when a table exceeds 512 tokens?** (~25-30% of FinanceBench evidence does — see §2.3).

Three standard techniques exist:

| Technique | How | Trade-off |
|---|---|---|
| **a) Whole table as 1 chunk** | Bump chunk size for that table. BGE-M3 accepts 8192, OpenAI 8191 | Uneven chunk sizes |
| **b) Header-repetition** | Split rows, repeat the column header in each chunk | Simple, preserves 512 fixed; duplicates tokens |
| **c) Row-as-sentence** | Convert each row to prose: *"In 2018 Net sales were $32,765M; ..."* | Reads naturally to dense embedders; loses precision if conversion fails |

The baseline picks **header-repetition (b)**. Reasons: clean contrast against the other 3 chunking strategies (semantic, contextual, late chunking) and trivial implementation. Let's code it on the real 3M Income Statement.


In [3]:
# === Block 1: header-repetition implementation ===
def chunk_table_with_header_repetition(table, header_row_idx=0, max_rows_per_chunk=10):
    """Split a table into chunks repeating the header row in each.

    Why this preserves info: the row ↔ column association lives in the HEADER
    (it tells you which column is "2018", "2017", "2016"). If we split the table
    by rows without repeating the header, downstream chunks become orphaned
    matrices of numbers with no column labels — the embedder has no semantic
    anchor to associate "32,765" with "Net sales 2018".

    Args:
        table: 2D list of cells (rows × cols).
        header_row_idx: which row carries the column header (typically 0).
        max_rows_per_chunk: data rows per chunk (excluding the repeated header).

    Returns:
        List of sub-tables, each starting with the header followed by data rows.
    """
    # Extract the header row (assumed to be at index 0 by default).
    header = table[header_row_idx]

    # Filter out the header from data rows: list comprehension with index check
    # is O(n) and avoids mutating the original table — safer than .pop().
    data_rows = [r for i, r in enumerate(table) if i != header_row_idx]

    chunks = []
    # range(0, n, step) gives us the start of each chunk window. Slicing
    # data_rows[i : i+step] handles the last partial chunk gracefully (no need
    # to special-case the tail).
    for i in range(0, len(data_rows), max_rows_per_chunk):
        # The actual chunk is [header] + slice → header is duplicated in EACH chunk.
        # That's the "repetition" — preserves the row ↔ column anchor at chunk boundaries.
        chunk = [header] + data_rows[i : i + max_rows_per_chunk]
        chunks.append(chunk)
    return chunks


# === Block 2: apply to the cleanly-extracted Income Statement ===
# We re-extract with the project-default settings (text-based strategy) to get
# the single clean 33×10 matrix. tables[0] picks the first (and only) table.
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[INCOME_STMT_IDX]
    settings = {"vertical_strategy": "text", "horizontal_strategy": "text"}
    income_table = page.extract_tables(settings)[0]

print(f"Original table: {len(income_table)} rows × {len(income_table[0])} cols")

# max_rows_per_chunk=10 is small for the demo so we get 3-4 visible chunks.
# In production, this value depends on token budget per chunk — for 512-token
# chunks with overlap=50, ~10-15 rows of a 10-col financial table fits comfortably.
chunks = chunk_table_with_header_repetition(
    income_table, header_row_idx=0, max_rows_per_chunk=10
)
print(f"\nGenerated {len(chunks)} chunks via header-repetition:\n")

# Iterate chunks and preview the first 4 rows of each (header + 3 data rows).
# Notice that EVERY chunk starts with the same header row — that's the strategy.
for i, chunk in enumerate(chunks):
    print(f"--- Chunk #{i + 1} ({len(chunk)} rows including repeated header) ---")
    for row in chunk[:4]:
        print(f"  {row}")
    if len(chunk) > 4:
        print(f"  ... ({len(chunk) - 4} more data rows)")
    print()


Original table: 33 rows × 10 cols

Generated 4 chunks via header-repetition:

--- Chunk #1 (11 rows including repeated header) ---
  ['Table of Contents', '', '', '', '', '', '', '', '', '']
  ['', '', '', '', '', '', '', '', '', '']
  ['3M Company and', 'Subsidiari', 'es', '', '', '', '', '', '', '']
  ['Consolidated Stat', 'ement of In', 'com e', '', '', '', '', '', '', '']
  ... (7 more data rows)

--- Chunk #2 (11 rows including repeated header) ---
  ['Table of Contents', '', '', '', '', '', '', '', '', '']
  ['Research, devel', 'opment and', 'related expenses', '', '', '1,821', '', '1,870', '', '1,764']
  ['Gain on sale of', 'businesses', '', '', '', '(547)', '', '(586)', '', '(111']
  ['Total operatin', 'g expenses', '', '', '', '25,558', '', '23,965', '', '23,082']
  ... (7 more data rows)

--- Chunk #3 (11 rows including repeated header) ---
  ['Table of Contents', '', '', '', '', '', '', '', '', '']
  ['', '', '', '', '', '', '', '', '', '']
  ['Less: Net income a', 'ttributa

### 3.3 The output JSONL schema — 1 line = 1 page

After parsing all 84 PDFs, we persist results in `data/processed/parsed/<doc_name>.jsonl` (one JSONL file per PDF). Each line represents a single page with the **minimal schema**:

```json
{
  "doc_name": "3M_2018_10K",
  "page_num": 56,
  "text": "Item 8. Financial Statements...",
  "tables": [[["Net sales", "", "$", "32,765", ...]]]
}
```

This format is streamable, programmatically accessible, and compatible with HuggingFace `datasets.load_dataset("json", ...)`. Recall from sub-block 2 that `evidence_page_num` is **singular** — that's what validates 1 line = 1 page as the right granularity.

Let's emit a single page record as a sanity check.


In [4]:
# === Block 1: canonical record builder ===
import json


def parse_page_to_record(pdf_path, doc_name, page_idx):
    """Parse a single page into the canonical JSONL record.

    The record is a dict with 4 keys exactly — no more, no less. We resist the
    urge to add bbox, char_count, or font metadata: extras get added ONLY when
    downstream metrics demand them. Premature schema bloat = harder to iterate
    later (every consumer needs to handle every field).
    """
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_idx]
        # Project-default table_settings — applied uniformly across the corpus.
        # Keep this in sync with scripts/parse_pdfs.py (single source of truth).
        settings = {"vertical_strategy": "text", "horizontal_strategy": "text"}
        return {
            "doc_name": doc_name,
            "page_num": page.page_number,                       # 1-indexed
            "text": page.extract_text() or "",                  # '' fallback if page is empty/image-only
            "tables": page.extract_tables(settings),            # list of List[List[str]]
        }


# === Block 2: emit one page record as a sanity check ===
# Same doc + page we've been using throughout §3 — Income Statement of 3M 2018.
record = parse_page_to_record(
    PDF_PATH, doc_name="3M_2018_10K", page_idx=INCOME_STMT_IDX
)

# Defensive print — verifies each field is populated and the shapes match
# what we computed in §3.1 (text_len ~1240, 1 table 33×10).
print(f"doc_name:      {record['doc_name']}")
print(f"page_num:      {record['page_num']}")
print(f"text length:   {len(record['text'])} chars")
print(f"# tables:      {len(record['tables'])}")
if record["tables"]:
    print(f"Table[0]:      {len(record['tables'][0])} rows × {len(record['tables'][0][0])} cols")

# === Block 3: serialize to a single JSONL line — exactly what hits disk ===
# json.dumps(..., ensure_ascii=False) preserves Unicode (ligatures, accents)
# instead of escaping to \uXXXX. Saves ~15-20% file size and stays human-readable.
# In the production batch, scripts/parse_pdfs.py writes one such line per page.
jsonl_line = json.dumps(record, ensure_ascii=False)
print(f"\nJSONL line length: {len(jsonl_line):,} chars")
print(f"Preview (first 220 chars):\n{jsonl_line[:220]}...")


doc_name:      3M_2018_10K
page_num:      56
text length:   1240 chars
# tables:      1
Table[0]:      33 rows × 10 cols

JSONL line length: 3,845 chars
Preview (first 220 chars):
{"doc_name": "3M_2018_10K", "page_num": 56, "text": "Table of Contents\n3M Company and Subsidiaries\nConsolidated Statement of Incom e\nYears ended December 31\n(Millions, except per share amounts) 2018 2017 2016\nNet sa...


### 3.4 Findings & decisions materialized

The 5 technical decisions of sub-block 4 are now demonstrated live:

| # | Decision | Demonstrated in |
|---|---|---|
| 1 | Library: `pdfplumber` | §3.0–§3.3 (text, tables, glyphs) |
| 2 | Table strategy: header-repetition | §3.2 — `chunk_table_with_header_repetition()` |
| 3 | Output: JSONL (1 line = 1 page) | §3.3 — `parse_page_to_record()` |
| 4 | Schema: `{doc_name, page_num, text, tables}` | §3.3 — record dict |
| 5 | Execution: serial | `scripts/parse_pdfs.py` (production batch) |

**Gotcha surfaced live in §3.1**: `extract_tables()` with default settings **fragments the Income Statement** into ~12 sub-tables because of horizontal separator lines between rows. The fix is `{"vertical_strategy": "text", "horizontal_strategy": "text"}` — yields 1 clean 33×10 matrix. This becomes the **project-default table_settings**.

**Next**: the production script `scripts/parse_pdfs.py` runs this same logic over all 84 PDFs serially and persists JSONL to `data/processed/parsed/`. Sub-block 5 will then build the chunking layer on top of the parsed corpus.

---


## 4. Building the indexable corpus — chunking with `tiktoken`

> **Goal of §4**: take the parsed JSONL produced in §3 (1 line per page) and turn it into the **indexable corpus** (1 line per chunk) that the embedders will ingest. We materialize the 5 technical decisions of sub-block 5 with executable demos against real data.

The full conceptual depth lives in `docs/CONTEXT.md` and the Notion descriptive memory of sub-block 5. Here we **make the chunking decisions concrete** with cells you can run end-to-end.


### 4.1 Tokenizer — `tiktoken` cl100k_base as project reference

The first decision: what counts as "512 tokens"? Different embedders use different tokenizers (OpenAI cl100k_base, BGE-M3 XLM-R, Voyage proprietary). The baseline picks **a single reference tokenizer** (`tiktoken cl100k_base`) and lets the others approximate. Three reasons:

1. All 5 target embedders accept ≥8192 token contexts → chunking to 512 *real* cl100k tokens never exceeds any embedder's window, even if their tokenizer measures ~10% more.
2. Standard approach in RAG literature.
3. Single tokenizer = single corpus = clean comparison between embedders.

Setup is a one-liner; the encoder is the same one OpenAI uses internally.


In [ ]:
# === Block 1: Load the tiktoken encoder and define the helper ===
# tiktoken is OpenAI's official tokenizer library (open-source).
# cl100k_base is the encoding used by GPT-3.5/GPT-4/text-embedding-3-*.
# We use it as the *single reference* for token counting across the project,
# even though we'll evaluate 5 different embedders downstream.
import tiktoken

ENCODING_NAME = "cl100k_base"
enc = tiktoken.get_encoding(ENCODING_NAME)

def count_tokens(text: str) -> int:
    """Token count with the project-reference encoder.

    Use this everywhere we need to enforce a token budget — never mix with
    char-based heuristics (chars/4) which can drift by 15-20% on financial
    text with many short numeric tokens like "32,765".
    """
    return len(enc.encode(text))


# Sanity check — encode a real fragment from the 3M Income Statement
sample = "Net sales $ 32,765 $ 31,657 $ 30,109"
print(f"Sample: {sample!r}")
print(f"Encoded:  {enc.encode(sample)}")
print(f"Token count: {count_tokens(sample)}")


### 4.2 Pre-filtering false-positive tables — the numeric-column heuristic

`pdfplumber` with `text-based` strategy detects ANY columnar structure as a "table" — including cover pages, page headers, and tab-separated lists. The naive filter (≥3 rows × ≥2 cols) is **not enough**: 3M's cover page parses as a 70×13 "table" with the same density (~35%) as the real Income Statement (33×10).

The discriminating heuristic: **at least 1 column where ≥30% of the cells are numeric** (matching `^[\d,.()$%-]+$`). Real financial tables have at least one numeric column; cover-page artifacts and header blocks do not. This rule cut the chunk count of 3M from 1,029 → 502 (~50% noise reduction).


In [ ]:
# === Block 1: Define the numeric-column filter ===
import re
import json

NUMERIC_RE = re.compile(r"^[\d,.()$%\s+-]+$")
NUMERIC_COL_RATIO = 0.3
MIN_TABLE_ROWS = 3
MIN_TABLE_COLS = 2


def is_numeric_cell(cell) -> bool:
    """A cell is numeric if it matches digits + standard financial separators."""
    if cell is None:
        return False
    s = str(cell).strip()
    return bool(s) and bool(NUMERIC_RE.match(s))


def has_numeric_column(table) -> bool:
    """True if at least one column has ≥NUMERIC_COL_RATIO numeric cells.

    This is the empirical discriminator. Density alone (% non-empty cells) does
    not separate real tables from cover-page artifacts — both sit ~35%. Numeric-
    column presence does, because financial tables have at least one column of
    numbers and cover pages have none.
    """
    if not table:
        return False
    n_rows = len(table)
    n_cols = max(len(r) for r in table)
    for col_idx in range(n_cols):
        col = [row[col_idx] if col_idx < len(row) else None for row in table]
        if sum(1 for c in col if is_numeric_cell(c)) / n_rows >= NUMERIC_COL_RATIO:
            return True
    return False


def is_real_table(table) -> bool:
    """The combined filter: shape check + numeric-column check."""
    if not table or len(table) < MIN_TABLE_ROWS:
        return False
    if len(table[0]) < MIN_TABLE_COLS:
        return False
    return has_numeric_column(table)


# === Block 2: Demo the filter on parsed pages of 3M 2018 ===
# Compare the cover page (page 1) vs the real Income Statement (page 56).
# Both pass the basic shape check; only the IS passes the numeric-column check.
PARSED_PATH = "../data/processed/parsed/3M_2018_10K.jsonl"

print(f"{'Page':>5} | {'Shape':>9} | {'has_numeric_col':>16} | Verdict")
print("-" * 60)
with open(PARSED_PATH) as f:
    for line in f:
        page = json.loads(line)
        if page["page_num"] not in [1, 56]:
            continue
        for table in page.get("tables", []):
            if not table:
                continue
            shape = f"{len(table):>3}x{len(table[0]):>2}"
            has_num = has_numeric_column(table)
            verdict = "REAL ✓" if is_real_table(table) else "FALSE-POSITIVE ✗"
            print(f"{page['page_num']:>5} | {shape:>9} | {str(has_num):>16} | {verdict}")


### 4.3 Chunking text — token-based windows with overlap

For text content (narrative sections like Item 1A, MD&A, Notes prose), we slide a window of `CHUNK_SIZE` tokens with a stride of `CHUNK_SIZE - OVERLAP`. The 50-token overlap prevents concepts from being cut at chunk boundaries.

The implementation is straightforward but **must use the reference tokenizer** — never count tokens with char-based approximations or you'll drift past the embedder's context window on long pages.


In [ ]:
# === Block 1: chunk_text — sliding window with overlap ===
CHUNK_SIZE = 512
OVERLAP = 50


def chunk_text(text: str) -> list[str]:
    """Split text into overlapping token-based chunks.

    If the text fits in one chunk → return [text].
    Otherwise → emit windows of CHUNK_SIZE tokens with stride CHUNK_SIZE-OVERLAP.

    The decode step turns token IDs back into text. Round-trip can introduce
    minor whitespace artifacts (rare with cl100k_base on English) — acceptable
    for embedding purposes.
    """
    tokens = enc.encode(text)
    if len(tokens) <= CHUNK_SIZE:
        return [text]

    stride = CHUNK_SIZE - OVERLAP
    chunks: list[str] = []
    i = 0
    while i < len(tokens):
        window = tokens[i : i + CHUNK_SIZE]
        chunks.append(enc.decode(window))
        if i + CHUNK_SIZE >= len(tokens):
            break
        i += stride
    return chunks


# === Block 2: Demo on a long narrative page ===
# Load page 5 of 3M 2018 (likely Item 1 — Business description, narrative-heavy).
# Show the chunking output and verify the overlap.
with open(PARSED_PATH) as f:
    pages = [json.loads(line) for line in f]

long_page = max((p for p in pages if p["page_num"] <= 30), key=lambda p: len(p["text"]))
print(f"Page {long_page['page_num']}: {len(long_page['text']):,} chars, "
      f"{count_tokens(long_page['text']):,} tokens")

text_chunks = chunk_text(long_page["text"])
print(f"Produced {len(text_chunks)} chunks:")
for i, chunk in enumerate(text_chunks):
    n_tok = count_tokens(chunk)
    print(f"  Chunk {i + 1}: {n_tok} tokens")

# Verify overlap empirically: last 30 chars of chunk N should appear in chunk N+1
if len(text_chunks) >= 2:
    tail_of_first = text_chunks[0][-80:]
    head_of_second = text_chunks[1][:160]
    overlap_in_second = tail_of_first[-30:] in head_of_second
    print(f"\n[overlap verification] last fragment of chunk 1 appears in chunk 2: {overlap_in_second}")


### 4.4 Chunking tables — header-repetition + threshold

For table content, we apply the **header-repetition** strategy from §3 — but with a key addition: **threshold-based activation**. A small table that fits in ≤512 tokens is emitted as a single chunk; only tables exceeding 512 tokens trigger the row-splitting logic.

Why threshold-based? Most financial tables are small (Income Statement: 33 rows, fits in ~480 tokens). Splitting them artificially would fragment information that's already self-contained. Header-repetition is reserved for the ~5-10% of tables that genuinely exceed the budget (typical: detailed Notes tables on stock-based compensation, segment reporting, leases).


In [ ]:
# === Block 1: Render a table as a markdown-style string ===
def table_to_markdown(table) -> str:
    """Convert a 2D table to a pipe-separated string.

    Empty cells become "" (preserves layout). Embedders read this as
    structured tabular text — close enough to genuine markdown that the
    semantic gap is negligible at retrieval time.
    """
    return "\n".join(
        " | ".join(str(c) if c is not None else "" for c in row)
        for row in table
    )


# === Block 2: chunk_table — threshold-based header-repetition ===
def chunk_table(table) -> list[str]:
    """Apply header-repetition only when the table exceeds CHUNK_SIZE tokens.

    Path A (small table): emit as single chunk, preserves all structure.
    Path B (large table): split rows greedily under (CHUNK_SIZE - header_tokens)
                          budget, prepend the header to every sub-chunk.
    """
    full_text = table_to_markdown(table)
    if count_tokens(full_text) <= CHUNK_SIZE:
        return [full_text]  # Path A

    # Path B: header-repetition
    header = table[0]
    data_rows = table[1:]
    header_text = table_to_markdown([header])
    budget_for_data = CHUNK_SIZE - count_tokens(header_text)

    chunks: list[str] = []
    current_rows: list = []
    current_tokens = 0
    for row in data_rows:
        row_tokens = count_tokens(table_to_markdown([row])) + 1  # +1 for the joining newline
        if current_rows and current_tokens + row_tokens > budget_for_data:
            chunks.append(table_to_markdown([header] + current_rows))
            current_rows = [row]
            current_tokens = row_tokens
        else:
            current_rows.append(row)
            current_tokens += row_tokens
    if current_rows:
        chunks.append(table_to_markdown([header] + current_rows))
    return chunks


# === Block 3: Demo on the 3M Income Statement (page 56) ===
# This table fits in ~480 tokens after rendering — Path A applies (single chunk).
income_page = next(p for p in pages if p["page_num"] == 56)
income_table = income_page["tables"][0]

print(f"Income Statement: {len(income_table)} rows × {len(income_table[0])} cols")
print(f"Rendered token count: {count_tokens(table_to_markdown(income_table))}")
print(f"Chunks produced: {len(chunk_table(income_table))} (Path A — fits in 1 chunk)\n")

# To demo Path B (header-repetition), we artificially shrink CHUNK_SIZE for the demo.
# In production the threshold is 512; here we use 200 to force splitting.
def chunk_table_demo(table, demo_chunk_size=200):
    """Same as chunk_table but with a custom small budget for demo purposes."""
    full = table_to_markdown(table)
    if count_tokens(full) <= demo_chunk_size:
        return [full]
    header = table[0]
    data_rows = table[1:]
    budget = demo_chunk_size - count_tokens(table_to_markdown([header]))
    out, cur, ctok = [], [], 0
    for row in data_rows:
        rtok = count_tokens(table_to_markdown([row])) + 1
        if cur and ctok + rtok > budget:
            out.append(table_to_markdown([header] + cur))
            cur, ctok = [row], rtok
        else:
            cur.append(row); ctok += rtok
    if cur:
        out.append(table_to_markdown([header] + cur))
    return out


demo_chunks = chunk_table_demo(income_table, demo_chunk_size=200)
print(f"Demo (CHUNK_SIZE=200, forces Path B): {len(demo_chunks)} chunks via header-repetition")
for i, c in enumerate(demo_chunks):
    print(f"  Chunk {i + 1}: {count_tokens(c)} tokens, starts with: {c[:80].replace(chr(10), ' / ')!r}")


### 4.5 The full corpus — aggregate stats over 84 PDFs

All the per-page logic above is bundled in `scripts/build_corpus.py`. After running it on all 84 PDFs (~7 seconds serial), we have the indexable corpus in `data/processed/chunks/`. Let's read it back and validate the totals.

These numbers are the **quantitative grounding** that converts the experimental plan from "feasible-on-paper" to "feasible-with-evidence" — the type of figure a Senior AI Engineer interview will explicitly ask about.


In [ ]:
# === Block 1: Aggregate stats over the chunked corpus ===
import statistics as stats
from pathlib import Path
from collections import Counter

CHUNKS_DIR = Path("../data/processed/chunks")

all_chunks = []
chunks_per_doc = Counter()
for f in CHUNKS_DIR.glob("*.jsonl"):
    with open(f) as fh:
        for line in fh:
            all_chunks.append(json.loads(line))
            chunks_per_doc[f.stem] += 1

text = [c for c in all_chunks if c["chunk_type"] == "text"]
table = [c for c in all_chunks if c["chunk_type"] == "table"]
text_tokens = [c["n_tokens"] for c in text]
table_tokens = [c["n_tokens"] for c in table]
total_tokens = sum(c["n_tokens"] for c in all_chunks)

print("=" * 60)
print("CORPUS-LEVEL STATS")
print("=" * 60)
print(f"Total chunks:        {len(all_chunks):>10,}")
print(f"  Text chunks:       {len(text):>10,}  ({len(text) / len(all_chunks) * 100:.1f}%)")
print(f"  Table chunks:      {len(table):>10,}  ({len(table) / len(all_chunks) * 100:.1f}%)")
print()
print(f"Token distribution:")
print(f"  Text:   median={int(stats.median(text_tokens))}, "
      f"p95={int(stats.quantiles(text_tokens, n=20)[18])}, max={max(text_tokens)}")
print(f"  Table:  median={int(stats.median(table_tokens))}, "
      f"p95={int(stats.quantiles(table_tokens, n=20)[18])}, max={max(table_tokens)}")
print()
print(f"Chunks per doc:")
counts = list(chunks_per_doc.values())
print(f"  min={min(counts)}, median={int(stats.median(counts))}, max={max(counts)}, mean={int(stats.mean(counts))}")
top3 = chunks_per_doc.most_common(3)
print(f"  Top 3 docs: {top3}")
print()
print(f"=== Embedding cost projection ===")
print(f"Total tokens (real, tiktoken):   {total_tokens:>12,}")
print(f"OpenAI text-embedding-3-large @ $0.13/1M:  ${total_tokens * 0.13 / 1_000_000:.2f} per pass")
print(f"Full experiment (5 embedders × 4 chunking, ~50% chunk reuse):  ~$15-30")
print(f"Project budget: ~$40-50 USD — fits comfortably ✓")
